# Data Encryption Standard in Python

DES encrypts a 64 bit plaintext block with a 64 bit key. This notebook follows Key Generation, the round function, encryption, and decryption.

## Hexadecimal and binary conversion

A value with 16 hexadecimal digits contains 64 bits. Conversion to binary keeps all 64 bits, and conversion back to hexadecimal keeps all 16 digits.

In [1]:
def hex_to_binary(hex_value):
    return format(int(hex_value, 16), "064b")


def binary_to_hex(binary_value):
    return format(int(binary_value, 2), "016X")

## Permutation, XOR addition, and Left Shift

A permutation rearranges bit positions. XOR addition combines bits. Left Shift moves the two key halves.

In [2]:
def permute(bits, table):
    result = ""
    for position in table:
        result += bits[position - 1]
    return result


def xor_bits(first, second):
    result = ""
    for index in range(len(first)):
        result += str(int(first[index]) ^ int(second[index]))
    return result


def left_shift(bits, amount):
    return bits[amount:] + bits[:amount]

## Phase 1: Key Generation

PC-1 selects 56 bits from the original key. The shifts move C and D. PC-2 selects 48 bits for each round subkey.

In [3]:
PC_1 = [
    57, 49, 41, 33, 25, 17, 9,
    1, 58, 50, 42, 34, 26, 18,
    10, 2, 59, 51, 43, 35, 27,
    19, 11, 3, 60, 52, 44, 36,
    63, 55, 47, 39, 31, 23, 15,
    7, 62, 54, 46, 38, 30, 22,
    14, 6, 61, 53, 45, 37, 29,
    21, 13, 5, 28, 20, 12, 4,
]

SHIFTS = [1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1]

PC_2 = [
    14, 17, 11, 24, 1, 5,
    3, 28, 15, 6, 21, 10,
    23, 19, 12, 4, 26, 8,
    16, 7, 27, 20, 13, 2,
    41, 52, 31, 37, 47, 55,
    30, 40, 51, 45, 33, 48,
    44, 49, 39, 56, 34, 53,
    46, 42, 50, 36, 29, 32,
]

## Initial Permutation IP of M and E Bit Selection

IP rearranges the 64 bit block. E Bit Selection expands the right half from 32 bits to 48 bits.

In [4]:
IP = [
    58, 50, 42, 34, 26, 18, 10, 2,
    60, 52, 44, 36, 28, 20, 12, 4,
    62, 54, 46, 38, 30, 22, 14, 6,
    64, 56, 48, 40, 32, 24, 16, 8,
    57, 49, 41, 33, 25, 17, 9, 1,
    59, 51, 43, 35, 27, 19, 11, 3,
    61, 53, 45, 37, 29, 21, 13, 5,
    63, 55, 47, 39, 31, 23, 15, 7,
]

E = [
    32, 1, 2, 3, 4, 5,
    4, 5, 6, 7, 8, 9,
    8, 9, 10, 11, 12, 13,
    12, 13, 14, 15, 16, 17,
    16, 17, 18, 19, 20, 21,
    20, 21, 22, 23, 24, 25,
    24, 25, 26, 27, 28, 29,
    28, 29, 30, 31, 32, 1,
]

## Round function f: S boxes

Each S box changes six bits to four bits. The first and last bits determine the row, and the middle four bits determine the column.

In [5]:
S_BOXES = [
    [
        [14, 4, 13, 1, 2, 15, 11, 8, 3, 10, 6, 12, 5, 9, 0, 7],
        [0, 15, 7, 4, 14, 2, 13, 1, 10, 6, 12, 11, 9, 5, 3, 8],
        [4, 1, 14, 8, 13, 6, 2, 11, 15, 12, 9, 7, 3, 10, 5, 0],
        [15, 12, 8, 2, 4, 9, 1, 7, 5, 11, 3, 14, 10, 0, 6, 13],
    ],
    [
        [15, 1, 8, 14, 6, 11, 3, 4, 9, 7, 2, 13, 12, 0, 5, 10],
        [3, 13, 4, 7, 15, 2, 8, 14, 12, 0, 1, 10, 6, 9, 11, 5],
        [0, 14, 7, 11, 10, 4, 13, 1, 5, 8, 12, 6, 9, 3, 2, 15],
        [13, 8, 10, 1, 3, 15, 4, 2, 11, 6, 7, 12, 0, 5, 14, 9],
    ],
    [
        [10, 0, 9, 14, 6, 3, 15, 5, 1, 13, 12, 7, 11, 4, 2, 8],
        [13, 7, 0, 9, 3, 4, 6, 10, 2, 8, 5, 14, 12, 11, 15, 1],
        [13, 6, 4, 9, 8, 15, 3, 0, 11, 1, 2, 12, 5, 10, 14, 7],
        [1, 10, 13, 0, 6, 9, 8, 7, 4, 15, 14, 3, 11, 5, 2, 12],
    ],
    [
        [7, 13, 14, 3, 0, 6, 9, 10, 1, 2, 8, 5, 11, 12, 4, 15],
        [13, 8, 11, 5, 6, 15, 0, 3, 4, 7, 2, 12, 1, 10, 14, 9],
        [10, 6, 9, 0, 12, 11, 7, 13, 15, 1, 3, 14, 5, 2, 8, 4],
        [3, 15, 0, 6, 10, 1, 13, 8, 9, 4, 5, 11, 12, 7, 2, 14],
    ],
    [
        [2, 12, 4, 1, 7, 10, 11, 6, 8, 5, 3, 15, 13, 0, 14, 9],
        [14, 11, 2, 12, 4, 7, 13, 1, 5, 0, 15, 10, 3, 9, 8, 6],
        [4, 2, 1, 11, 10, 13, 7, 8, 15, 9, 12, 5, 6, 3, 0, 14],
        [11, 8, 12, 7, 1, 14, 2, 13, 6, 15, 0, 9, 10, 4, 5, 3],
    ],
    [
        [12, 1, 10, 15, 9, 2, 6, 8, 0, 13, 3, 4, 14, 7, 5, 11],
        [10, 15, 4, 2, 7, 12, 9, 5, 6, 1, 13, 14, 0, 11, 3, 8],
        [9, 14, 15, 5, 2, 8, 12, 3, 7, 0, 4, 10, 1, 13, 11, 6],
        [4, 3, 2, 12, 9, 5, 15, 10, 11, 14, 1, 7, 6, 0, 8, 13],
    ],
    [
        [4, 11, 2, 14, 15, 0, 8, 13, 3, 12, 9, 7, 5, 10, 6, 1],
        [13, 0, 11, 7, 4, 9, 1, 10, 14, 3, 5, 12, 2, 15, 8, 6],
        [1, 4, 11, 13, 12, 3, 7, 14, 10, 15, 6, 8, 0, 5, 9, 2],
        [6, 11, 13, 8, 1, 4, 10, 7, 9, 5, 0, 15, 14, 2, 3, 12],
    ],
    [
        [13, 2, 8, 4, 6, 15, 11, 1, 10, 9, 3, 14, 5, 0, 12, 7],
        [1, 15, 13, 8, 10, 3, 7, 4, 12, 5, 6, 11, 0, 14, 9, 2],
        [7, 11, 4, 1, 9, 12, 14, 2, 0, 6, 10, 13, 15, 3, 5, 8],
        [2, 1, 14, 7, 4, 10, 8, 13, 15, 12, 9, 0, 3, 5, 6, 11],
    ],
]

## P and final permutation

P permutes the result of the S boxes. The inverse of IP is applied after 16 rounds.

In [6]:
P = [
    16, 7, 20, 21,
    29, 12, 28, 17,
    1, 15, 23, 26,
    5, 18, 31, 10,
    2, 8, 24, 14,
    32, 27, 3, 9,
    19, 13, 30, 6,
    22, 11, 4, 25,
]

IP_INVERSE = [
    40, 8, 48, 16, 56, 24, 64, 32,
    39, 7, 47, 15, 55, 23, 63, 31,
    38, 6, 46, 14, 54, 22, 62, 30,
    37, 5, 45, 13, 53, 21, 61, 29,
    36, 4, 44, 12, 52, 20, 60, 28,
    35, 3, 43, 11, 51, 19, 59, 27,
    34, 2, 42, 10, 50, 18, 58, 26,
    33, 1, 41, 9, 49, 17, 57, 25,
]

## Generating 16 Subkeys

The 56 bit key is split into C0 and D0. Each round shifts both halves and applies PC-2 to form a 48 bit subkey.

In [7]:
def generate_round_keys(key):
    key_bits = hex_to_binary(key)
    selected_key = permute(key_bits, PC_1)
    left = selected_key[:28]
    right = selected_key[28:]
    round_keys = []

    for amount in SHIFTS:
        left = left_shift(left, amount)
        right = left_shift(right, amount)
        round_keys.append(permute(left + right, PC_2))

    return round_keys

## Round function f

The round function uses E Bit Selection, Key Mixing with XOR, the S boxes, and P.

In [8]:
def s_box_substitution(bits):
    result = ""

    for index in range(8):
        group = bits[index * 6:(index + 1) * 6]
        row = int(group[0] + group[5], 2)
        column = int(group[1:5], 2)
        value = S_BOXES[index][row][column]
        result += format(value, "04b")

    return result


def round_function(right, round_key):
    expanded = permute(right, E)
    mixed = xor_bits(expanded, round_key)
    substituted = s_box_substitution(mixed)
    return permute(substituted, P)

## Round Update Rule and Decryption

The new left half is the previous right half. The new right half is the previous left half XOR f. After 16 rounds, reverse the order of the blocks and apply the final permutation. Decryption uses the round subkeys in reverse order.

In [9]:
def des_process(block, round_keys):
    block_bits = hex_to_binary(block)
    initial_bits = permute(block_bits, IP)
    left = initial_bits[:32]
    right = initial_bits[32:]

    for round_key in round_keys:
        next_left = right
        next_right = xor_bits(left, round_function(right, round_key))
        left = next_left
        right = next_right

    final_bits = permute(right + left, IP_INVERSE)
    return binary_to_hex(final_bits)


def des_encrypt(plaintext, key):
    plaintext = require_hex(plaintext)
    key = require_hex(key)
    round_keys = generate_round_keys(key)
    return des_process(plaintext, round_keys)


def des_decrypt(ciphertext, key):
    ciphertext = require_hex(ciphertext)
    key = require_hex(key)
    round_keys = generate_round_keys(key)
    return des_process(ciphertext, round_keys[::-1])

## Program Input and Output

The key and plaintext must each contain exactly 16 hexadecimal digits. Lowercase letters are accepted. The program displays the plaintext, ciphertext, and decrypted plaintext.

In [10]:
def valid_hex(value):
    if not isinstance(value, str) or len(value) != 16:
        return False

    for character in value:
        if character not in "0123456789ABCDEFabcdef":
            return False

    return True


def require_hex(value):
    if not valid_hex(value):
        raise ValueError("Enter exactly 16 hexadecimal digits.")
    return value.upper()


def read_hex(prompt):
    while True:
        value = input(prompt)
        if valid_hex(value):
            return value.upper()
        print("Enter exactly 16 hexadecimal digits.")


def main():
    key = read_hex("Enter key (16 hex digits): ")
    plaintext = read_hex("Enter plaintext (16 hex digits): ")
    ciphertext = des_encrypt(plaintext, key)
    decrypted_plaintext = des_decrypt(ciphertext, key)

    print("Plaintext:", plaintext)
    print("Ciphertext:", ciphertext)
    print("Decrypted plaintext:", decrypted_plaintext)

## Known ciphertexts

These key and plaintext combinations check the ciphertext and recovered plaintext. The first combination is the lecture example.

In [11]:
cases = [
    ("133457799BBCDFF1", "0123456789ABCDEF", "85E813540F0AB405"),
    ("0000000000000000", "0000000000000000", "8CA64DE9C1B123A7"),
    ("FFFFFFFFFFFFFFFF", "FFFFFFFFFFFFFFFF", "7359B2163E4EDC58"),
]

for key, plaintext, expected_ciphertext in cases:
    ciphertext = des_encrypt(plaintext, key)
    decrypted_plaintext = des_decrypt(ciphertext, key)
    assert ciphertext == expected_ciphertext
    assert decrypted_plaintext == plaintext
    print("Key:", key)
    print("Plaintext:", plaintext)
    print("Ciphertext:", ciphertext)
    print("Decrypted plaintext:", decrypted_plaintext)
    print()

Key: 133457799BBCDFF1
Plaintext: 0123456789ABCDEF
Ciphertext: 85E813540F0AB405
Decrypted plaintext: 0123456789ABCDEF

Key: 0000000000000000
Plaintext: 0000000000000000
Ciphertext: 8CA64DE9C1B123A7
Decrypted plaintext: 0000000000000000

Key: FFFFFFFFFFFFFFFF
Plaintext: FFFFFFFFFFFFFFFF
Ciphertext: 7359B2163E4EDC58
Decrypted plaintext: FFFFFFFFFFFFFFFF

